## GPT-2 Autoregressive Generation

Welcome back to Harnessing Transformers with Hugging Face! As we embark on the third lesson of our transformative journey, we shift our focus from understanding to creation. You've witnessed BERT's remarkable ability to comprehend language through bidirectional attention and how it peers both forward and backward to grasp meaning with unprecedented depth. Now, prepare to explore an architecture that approaches language from an entirely different philosophy: GPT-2 (Generative Pre-trained Transformer 2), a model that thinks like a writer, not a reader.

Imagine the difference between analyzing a completed painting and creating one brushstroke by brushstroke: this captures the fundamental distinction between BERT and GPT-2. Where BERT's encoder-only design allows it to see the complete picture simultaneously, GPT-2's decoder-only architecture constrains it to work autoregressively, building text one token at a time using only what came before. This apparent limitation becomes GPT-2's greatest strength: by learning to predict what comes next based solely on preceding context, it develops an uncanny ability to generate human-like text that flows naturally and coherently. From creative storytelling to code completion, from dialogue generation to technical writing, GPT-2 has revolutionized how we think about machine-generated text. By lesson's end, you'll master GPT-2's causal attention mechanism, understand its sophisticated BPE tokenization, and wield various decoding strategies to control the creativity and quality of generated content.

---

### Understanding GPT-2's Autoregressive Architecture

GPT-2 represents a fundamentally different approach to language modeling compared to the bidirectional understanding we explored with BERT. At its core lies a decoder-only architecture that processes text autoregressively, a term that captures how the model uses its own previous predictions to generate subsequent ones. This design philosophy mirrors the human writing process: when we compose text, we consider what we've already written to determine what should come next, unable to peek ahead at words we haven't yet conceived.

The architectural constraint that defines GPT-2 is causal attention, also called masked self-attention in the decoder context. Unlike BERT's attention mechanism, where every token can attend to every other token in the sequence, GPT-2's attention is strictly causal: a token at position $i$ can only attend to tokens at positions $1$ through $i$, never to future positions $i+1$ and beyond. This is enforced through an upper-triangular attention mask that prevents the model from "cheating" by looking ahead during training. When GPT-2 learns to predict the word "cat" in the sentence "The small cat jumped," it can only use "The small" as context, not the subsequent "jumped", exactly as a human writer would when deciding what noun to place after "small."

This autoregressive constraint shapes everything about how GPT-2 learns and generates text. During training on massive text corpora, GPT-2 learns to maximize the probability of each token given all preceding tokens:

$$P(x_t \mid x_1, x_2, \ldots, x_{t-1})$$

This creates a model that excels at understanding the statistical patterns of language — not just grammar and syntax, but style, tone, and even factual associations. The result is a model capable of generating remarkably coherent and contextually appropriate text, from completing simple prompts to crafting entire articles, all by repeatedly answering the question: "Given everything written so far, what word most naturally comes next?"

---

### GPT-2's Byte Pair Encoding Tokenization

Before GPT-2 can generate its first word, it must first decompose text into manageable units through Byte Pair Encoding (BPE), which is a tokenization strategy specifically designed to balance vocabulary efficiency with the flexibility needed for open-ended text generation. Understanding BPE is crucial because it directly impacts what GPT-2 can generate and how naturally the generated text flows.

```python
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def explore_gpt2_tokenizer():
    """Explore GPT-2's BPE tokenization"""
    print("GPT-2 BPE Tokenization:")

    # Load the GPT-2 tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

    # Test tokenization scenarios
    test_texts = ["Hello world", "Tokenization", "Transformers"]

    for text in test_texts:
        # Tokenize the text to see subword units
        tokens = tokenizer.tokenize(text)
        print(f"'{text}' -> {tokens}")
```

The BPE tokenization results reveal GPT-2's distinctive approach to text decomposition:

```text
GPT-2 BPE Tokenization:
'Hello world' -> ['Hello', 'Ġworld']
'Tokenization' -> ['Token', 'ization']
'Transformers' -> ['Transform', 'ers']
```

These tokenization patterns showcase BPE's elegant solution to a fundamental challenge in language modeling. Notice the peculiar `Ġ` symbol in `['Hello', 'Ġworld']`: this represents a leading space, allowing GPT-2 to preserve exact spacing information within its vocabulary. This detail is critical for generation: GPT-2 must know whether to generate `world` (continuing a word) or `Ġworld` (starting a new word after a space). Without this distinction, generated text would either lack spaces between words or have unwanted spaces within words.

The subword decompositions like `['Token', 'ization']` and `['Transform', 'ers']` demonstrate BPE's morphological awareness. Built through statistical analysis of text frequency, BPE's vocabulary naturally captures common prefixes, suffixes, and word stems. This allows GPT-2 to generate virtually any word (even technical terms or neologisms never seen during training) by combining appropriate subword units. With a vocabulary of just 50,257 tokens, GPT-2 achieves the flexibility to generate any conceivable text while maintaining the efficiency needed for fast generation.

---

### Text Generation Through Autoregressive Decoding

Now let's witness GPT-2's defining capability in action: transforming a simple prompt into flowing, coherent text through autoregressive generation. This process reveals how GPT-2's architectural constraints become its generative power.

```python
import torch

def gpt2_generation_and_analysis():
    """Demonstrate GPT-2's generation and next token prediction"""
    print("GPT-2 Generation & Analysis:")

    # Load model and tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token  # Set padding token
    model.config.pad_token_id = tokenizer.pad_token_id  # Ensure model config matches tokenizer

    # Text generation
    prompt = "The future of AI is"
    inputs = tokenizer.encode(prompt, return_tensors="pt")

    with torch.no_grad():
        # Create attention mask for proper attention handling
        attention_mask = torch.ones_like(inputs)

        # Generate text with controlled randomness
        output = model.generate(
            inputs,
            attention_mask=attention_mask,
            max_length=50,
            do_sample=True,
            temperature=0.5,
            pad_token_id=tokenizer.eos_token_id
        )
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    print(f"Generated: '{generated_text}'")
```

The generation process exemplifies GPT-2's autoregressive nature in action:

```text
Generated: 'The future of AI is pretty much in the hands of humans.

But how does AI work?'
```

This generated text reveals the sophisticated process underlying GPT-2's generation. Starting with just the four tokens "The future of AI is", the model constructs a coherent continuation that demonstrates grammatical correctness, semantic relevance, and even philosophical depth. The generation unfolds token by token: after encoding the prompt, GPT-2 predicts the most likely next token (perhaps "pretty"), appends it to the sequence, then uses this extended context to predict the subsequent token, continuing this process up to 50 tokens or until generating an end-of-sequence marker.

The generation parameters reveal crucial controls over GPT-2's creative process. The `temperature=0.5` parameter acts as a "creativity dial": lower values (approaching 0) make generation more deterministic and focused on high-probability tokens, while higher values increase randomness and creative exploration. Setting `do_sample=True` enables probabilistic sampling from the token distribution rather than always selecting the highest-probability token, preventing the repetitive loops that can plague deterministic generation. These parameters let you tune GPT-2's output from conservative and predictable to wildly creative, adapting to different use cases from technical documentation to creative fiction.

---

### Analyzing Token Probabilities and Generation Decisions

To truly understand GPT-2's generation mechanism, let's peek under the hood at the probability distributions that guide each token selection. This analysis reveals the sophisticated reasoning that transforms statistical patterns into coherent text.

```python
import torch.nn.functional as F

def analyze_next_token_predictions():
    """Analyze GPT-2's next token prediction probabilities"""
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model = GPT2LMHeadModel.from_pretrained("gpt2")

    # Analyze next token predictions
    partial_sentence = "The weather today is"
    partial_inputs = tokenizer.encode(partial_sentence, return_tensors="pt")

    # Get model predictions
    outputs = model(partial_inputs)
    next_token_logits = outputs.logits[0, -1, :]  # Logits for next token

    # Convert to probabilities and get top predictions
    top_probs, top_indices = torch.topk(F.softmax(next_token_logits, dim=-1), 3)

    print(f"Next token predictions for '{partial_sentence}':")
    for prob, idx in zip(top_probs, top_indices):
        token = tokenizer.decode([idx])
        print(f"  '{token}': {prob.item():.3f}")

# Execute our analysis function
analyze_next_token_predictions()
```

The probability analysis unveils GPT-2's nuanced understanding of context:

```text
Next token predictions for 'The weather today is':
  ' very': 0.043
  ' good': 0.032
  ' pretty': 0.031
```

These probability distributions illuminate how GPT-2 transforms learned patterns into generation decisions. The top prediction `' very'` (4.3% probability) reflects GPT-2's understanding that weather descriptions often include intensity modifiers — a pattern learned from countless weather reports and conversations in its training data. The alternatives `' good'` (3.2%) and `' pretty'` (3.1%) represent equally valid but stylistically different continuations, showing GPT-2's awareness of multiple linguistic registers.

What's particularly revealing is the relatively low absolute probabilities (all under 5%) despite these being the top choices. This reflects the genuine uncertainty in natural language — after "The weather today is," dozens of continuations are plausible: "sunny," "terrible," "unpredictable," "perfect," and more. GPT-2's probability distribution captures this linguistic reality, spreading probability mass across many reasonable options rather than being overconfident. This uncertainty is precisely what makes sampling-based generation powerful: by drawing from this distribution probabilistically, GPT-2 can generate diverse, natural-sounding text that avoids the mechanical repetition of always choosing the single most likely token.

---

### Conclusion and Next Steps

You've now mastered GPT-2's autoregressive decoder architecture, discovering how its sequential, left-to-right processing creates a powerful engine for text generation. From BPE tokenization that elegantly handles any text while preserving crucial spacing information, to causal attention that enforces the fundamental constraint of only seeing past context, to sophisticated probability distributions that guide intelligent token selection — you've explored the complete pipeline that transforms simple prompts into flowing, coherent text. Your journey through both BERT's bidirectional understanding and GPT-2's autoregressive generation has given you a comprehensive view of the transformer landscape's two fundamental paradigms.

Armed with this deep understanding of decoder architectures and generation strategies, you're ready to harness GPT-2's creative power for real-world applications. The practice exercises ahead will cement your knowledge through hands-on implementation, letting you experiment with different generation strategies and experience firsthand how architectural choices translate into capabilities. As we approach the final lesson of our Hugging Face journey, you've built the foundation to understand and utilize the full spectrum of transformer architectures!


## GPT-2 Tokenization Pattern Analysis

After exploring GPT-2's architecture and tokenization concepts, it's time to get hands-on with BPE (Byte Pair Encoding) tokenization. You'll complete the `explore_gpt2_tokenizer()` function to analyze how GPT-2 breaks down different text inputs into subword tokens.

---

### Your Task

Your goal is to implement the missing parts by:

- Loading the GPT-2 tokenizer from Hugging Face
- Creating a list of test texts with different characteristics
- Tokenizing each text and displaying the results

The provided code already includes the function structure and a complete second function for reference. Focus on understanding how BPE tokenization handles various word patterns — this hands-on exploration will give you valuable insight into how GPT-2 processes text input!

---

### Starter Code

```python
from transformers import GPT2Tokenizer
import torch
import torch.nn.functional as F

def explore_gpt2_tokenizer():
    """Explore GPT-2's BPE tokenization"""
    print("GPT-2 BPE Tokenization:")

    # TODO: Load the GPT-2 tokenizer

    # TODO: Create a list of test texts to analyze

    # TODO: Loop through each text and tokenize it, then print the results



def main():
    explore_gpt2_tokenizer()


if __name__ == "__main__":
    main()
```


---

### Solution

Three TODOs needed code: loading the tokenizer, defining the test texts, and tokenizing and printing each one.

```python
# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Create a list of test texts to analyze
test_texts = ["Hello world", "Tokenization", "Transformers"]

# Loop through each text and tokenize it, then print the results
for text in test_texts:
    tokens = tokenizer.tokenize(text)
    print(f"'{text}' -> {tokens}")
```

- `GPT2Tokenizer.from_pretrained("gpt2")` loads GPT-2's byte-level BPE tokenizer with its 50,257-token vocabulary.
- `test_texts` is a plain list of strings. The task leaves the choice open, so this solution reuses the three texts from the lesson, which mix a two-word phrase and two longer words that split into subwords.
- `tokenizer.tokenize(text)` returns the list of subword strings. A leading space is encoded as `Ġ`, so `Hello world` becomes `['Hello', 'Ġworld']`.
- The tokenizer is created once, **before** the loop, so it is not reloaded for every text.
- Without the tokenizer line, `tokenizer` is undefined and the loop raises a `NameError`. Without `test_texts`, the `for` loop raises a `NameError` as well.
- The starter's `torch` and `torch.nn.functional` imports are unused in this function. They are left as given.

When the script runs, it prints the header and one `'<text>' -> [tokens]` line per text. With the lesson's three texts, the lesson shows `['Hello', 'Ġworld']`, `['Token', 'ization']` and `['Transform', 'ers']`. The first run downloads the tokenizer files, so it needs internet access.


In [ ]:
from transformers import GPT2Tokenizer
import torch
import torch.nn.functional as F

def explore_gpt2_tokenizer():
    """Explore GPT-2's BPE tokenization"""
    print("GPT-2 BPE Tokenization:")

    # Load the GPT-2 tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

    # Create a list of test texts to analyze
    test_texts = ["Hello world", "Tokenization", "Transformers"]

    # Loop through each text and tokenize it, then print the results
    for text in test_texts:
        tokens = tokenizer.tokenize(text)
        print(f"'{text}' -> {tokens}")



def main():
    explore_gpt2_tokenizer()


if __name__ == "__main__":
    main()


## Fixing GPT-2 Text Generation Issues

Now that you've explored how GPT-2 tokenizes text using BPE, it's time to work on the generation process itself. The provided code attempts to generate text using GPT-2, but there is something wrong with the generation parameters that will cause issues.

---

### Your Task

Your job is to run the code, observe what happens, and fix the parameter configuration in the `model.generate()` call. The generation function should produce high-quality text output with proper attention masking and controlled randomness.

Think about which parameters are needed for effective autoregressive text generation and how they impact the final output. Getting the generation parameters right is crucial for creating useful AI applications!

---

### Starter Code

```python
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import torch.nn.functional as F


def gpt2_generation_and_analysis():
    """Demonstrate GPT-2's generation and next token prediction"""
    print("GPT-2 Generation & Analysis:")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id  # Ensure model config matches tokenizer

    # Text generation
    prompt = "The future of AI is"
    inputs = tokenizer.encode(prompt, return_tensors="pt")

    with torch.no_grad():
        # Create attention mask
        attention_mask = torch.ones_like(inputs)

        # Generate text
        output = model.generate(inputs)
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

        # Next token prediction
        partial_sentence = "The weather today is"
        partial_inputs = tokenizer.encode(partial_sentence, return_tensors="pt")
        outputs = model(partial_inputs)
        next_token_logits = outputs.logits[0, -1, :]
        top_probs, top_indices = torch.topk(F.softmax(next_token_logits, dim=-1), 3)

    print(f"Generated: '{generated_text}'")
    print(f"Next token predictions for '{partial_sentence}':")
    for prob, idx in zip(top_probs, top_indices):
        token = tokenizer.decode([idx])
        print(f"  '{token}': {prob.item():.3f}")

def main():
    gpt2_generation_and_analysis()

if __name__ == "__main__":
    main()
```


---

### Solution

This exercise has no `# TODO`. The bug is the bare `model.generate(inputs)` call: it uses default settings and ignores the `attention_mask` that was created just above it. The fix is to pass the generation parameters explicitly.

```python
# Generate text with controlled randomness
output = model.generate(
    inputs,
    attention_mask=attention_mask,
    max_length=50,
    do_sample=True,
    temperature=0.5,
    pad_token_id=tokenizer.eos_token_id
)
```

- `attention_mask=attention_mask` passes the mask built in the line above, which marks every prompt token as real input. Without it, `attention_mask` is a variable that is created and never used, and `generate` has to guess the mask, which triggers a warning because the pad token equals the end-of-sequence token.
- `max_length=50` sets the total length, prompt included, so the output is not cut short at the library default.
- `do_sample=True` samples from the token distribution instead of always taking the single most likely token. The default is greedy decoding, which is deterministic and can fall into repetition.
- `temperature=0.5` sharpens the distribution, so the sampling stays focused on likely tokens. Lower values are more predictable and higher values are more random.
- `pad_token_id=tokenizer.eos_token_id` tells `generate` which ID to use for padding, which silences the missing-pad-token warning.
- `temperature` only has an effect when `do_sample=True`.

When the script runs, it prints a generated continuation of `The future of AI is`, then the top 3 next-token predictions for `The weather today is`. Because sampling is random, the generated text changes from run to run. The lesson shows `' very'`, `' good'` and `' pretty'` for the weather prompt. The first run downloads the GPT-2 model, so it needs internet access.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import torch.nn.functional as F


def gpt2_generation_and_analysis():
    """Demonstrate GPT-2's generation and next token prediction"""
    print("GPT-2 Generation & Analysis:")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id  # Ensure model config matches tokenizer

    # Text generation
    prompt = "The future of AI is"
    inputs = tokenizer.encode(prompt, return_tensors="pt")

    with torch.no_grad():
        # Create attention mask
        attention_mask = torch.ones_like(inputs)

        # Generate text with attention mask, length limit and controlled randomness
        output = model.generate(
            inputs,
            attention_mask=attention_mask,
            max_length=50,
            do_sample=True,
            temperature=0.5,
            pad_token_id=tokenizer.eos_token_id
        )
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

        # Next token prediction
        partial_sentence = "The weather today is"
        partial_inputs = tokenizer.encode(partial_sentence, return_tensors="pt")
        outputs = model(partial_inputs)
        next_token_logits = outputs.logits[0, -1, :]
        top_probs, top_indices = torch.topk(F.softmax(next_token_logits, dim=-1), 3)

    print(f"Generated: '{generated_text}'")
    print(f"Next token predictions for '{partial_sentence}':")
    for prob, idx in zip(top_probs, top_indices):
        token = tokenizer.decode([idx])
        print(f"  '{token}': {prob.item():.3f}")

def main():
    gpt2_generation_and_analysis()

if __name__ == "__main__":
    main()


## Inside GPT-2 Decision Making

## Creative versus Predictable Text Generation